In [ ]:
# Cell 1: Mount Google Drive, install dependencies, and set paths
# Run this once at the start of your Colab session.

import os
import sys
import subprocess

try:
    from google.colab import drive
except ImportError:
    raise RuntimeError("This notebook cell is intended for Google Colab (google.colab not found).")

# ==== 1. Mount Google Drive ====
print("=" * 80)
print("STEP 1: Mounting Google Drive")
print("=" * 80)

drive.mount("/content/drive")

# ==== 2. Define base paths (mirrors your local structure) ====
print("\n" + "=" * 80)
print("STEP 2: Setting base paths")
print("=" * 80)

BASE_DIR = "/content/drive/MyDrive/Libri_Vibevoice"
VIBEVOICE_DIR = os.path.join(BASE_DIR, "VibeVoice")
SPEAKERS_DIR = os.path.join(BASE_DIR, "speakers")

# You can customize where outputs are written by changing this path
OUTPUT_BASE_DIR = os.path.join(BASE_DIR, "outputs")  # <- change if you prefer a different location

print(f"BASE_DIR       : {BASE_DIR}")
print(f"VIBEVOICE_DIR  : {VIBEVOICE_DIR}")
print(f"SPEAKERS_DIR   : {SPEAKERS_DIR}")
print(f"OUTPUT_BASE_DIR: {OUTPUT_BASE_DIR}")

# Basic existence checks
if not os.path.isdir(VIBEVOICE_DIR):
    raise FileNotFoundError(f"VibeVoice directory not found at: {VIBEVOICE_DIR}")
if not os.path.isdir(SPEAKERS_DIR):
    raise FileNotFoundError(f"Speakers directory not found at: {SPEAKERS_DIR}")

# Optionally, quickly show how many top-level speaker folders we see
speaker_ids = [d for d in os.listdir(SPEAKERS_DIR) if os.path.isdir(os.path.join(SPEAKERS_DIR, d))]
print(f"Found {len(speaker_ids)} speaker folders: {sorted(speaker_ids)}")

os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

# ==== 3. Install Python dependencies (minimal, quiet) ====
print("\n" + "=" * 80)
print("STEP 3: Installing Python packages")
print("=" * 80)

packages = [
    "torch>=2.0.0",
    "torchaudio>=2.0.0",
    "soundfile>=0.12.0",
    "librosa>=0.10.0",
    "numpy>=1.24.0",
    "scipy>=1.10.0",
    "tqdm>=4.65.0",
    "transformers>=4.30.0",
    "sentencepiece>=0.1.99",
]

for package in packages:
    try:
        print(f"Installing {package}...")
        subprocess.run([sys.executable, "-m", "pip", "install", package, "--quiet"], check=True)
    except Exception as e:
        print(f"⚠ Warning: Failed to install {package}: {e}")

# ==== 4. Install VibeVoice from your Drive copy (editable) ====
print("\n" + "=" * 80)
print("STEP 4: Installing local VibeVoice package (editable mode)")
print("=" * 80)

if not os.path.isdir(VIBEVOICE_DIR):
    raise FileNotFoundError(f"VibeVoice directory not found at: {VIBEVOICE_DIR}")

# Remember original working directory so we can restore it later
original_cwd = os.getcwd()
os.chdir(VIBEVOICE_DIR)

try:
    # If there is a requirements file, install it first (quietly)
    if os.path.exists("requirements.txt"):
        print("Installing VibeVoice requirements...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--quiet"], check=False)

    print("Installing VibeVoice (editable mode)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"], check=True)
    print("✓ VibeVoice installed from Drive")
finally:
    os.chdir(original_cwd)

# Add VibeVoice repo to sys.path for direct imports if needed
if VIBEVOICE_DIR not in sys.path:
    sys.path.insert(0, VIBEVOICE_DIR)

# ==== 5. Quick GPU check ====
print("\n" + "=" * 80)
print("STEP 5: Checking GPU availability")
print("=" * 80)

import torch

if torch.cuda.is_available():
    print(f"✓ CUDA available: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA Version: {torch.version.cuda}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ Warning: CUDA not available. For best results, enable a GPU runtime in Colab.")

print("\nSetup complete. You can move on to the next cell.")



In [ ]:
# Cell 2: Load VibeVoice model and processor

import torch

from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

# You can change this if you want to try the Large model instead
MODEL_PATH = "vibevoice/VibeVoice-1.5B"

# Simple device selection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

print("Loading VibeVoice processor...")
processor = VibeVoiceProcessor.from_pretrained(MODEL_PATH)

# Decide dtype & attention implementation (mirrors demo scripts, but simplified)
if DEVICE == "cuda":
    load_dtype = torch.bfloat16
    attn_impl_primary = "flash_attention_2"
else:
    load_dtype = torch.float32
    attn_impl_primary = "sdpa"

print(
    f"Loading VibeVoice model from {MODEL_PATH} with dtype={load_dtype} and attn={attn_impl_primary}"
)

try:
    if DEVICE == "cuda":
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_PATH,
            torch_dtype=load_dtype,
            device_map="cuda",
            attn_implementation=attn_impl_primary,
        )
    else:
        # CPU fallback (much slower, but works for short tests)
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_PATH,
            torch_dtype=load_dtype,
            device_map="cpu",
            attn_implementation=attn_impl_primary,
        )
except Exception as e:
    # Fallback to SDPA if flash_attention_2 is not available
    if attn_impl_primary == "flash_attention_2":
        print(f"Error with flash_attention_2: {e}")
        print("Falling back to SDPA attention implementation.")
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_PATH,
            torch_dtype=load_dtype,
            device_map=("cuda" if DEVICE == "cuda" else "cpu"),
            attn_implementation="sdpa",
        )
    else:
        raise

model.eval()

# Inference settings
DDPM_STEPS = 10
model.set_ddpm_inference_steps(num_steps=DDPM_STEPS)

# Sample rate for saving/playing audio
SAMPLE_RATE = getattr(
    getattr(processor, "feature_extractor", None), "sampling_rate", 24000
)

print("Model and processor loaded successfully.")
print(f"DDPM steps: {DDPM_STEPS}, Sample rate: {SAMPLE_RATE}")



In [ ]:
# Cell 3: Scan LibriTTS-R speakers and pick longest reference wav per speaker

import os
from pathlib import Path
import soundfile as sf

# Mapping: speaker_id -> {"path": str, "duration_sec": float}
SPEAKER_REFERENCE_MAP = {}


def get_longest_wav_for_speaker(speaker_id: str):
    """Return (path, duration_sec) of the longest .wav for a given speaker.

    Speaker folder layout:
        SPEAKERS_DIR / speaker_id / <chapter_id> / *.wav
    """
    speaker_root = Path(SPEAKERS_DIR) / speaker_id
    if not speaker_root.is_dir():
        print(f"[WARN] Speaker folder not found: {speaker_root}")
        return None, 0.0

    wav_paths = list(speaker_root.rglob("*.wav"))
    if not wav_paths:
        print(f"[WARN] No .wav files found for speaker {speaker_id} under {speaker_root}")
        return None, 0.0

    best_path = None
    best_duration = 0.0

    for wav_path in wav_paths:
        try:
            audio, sr = sf.read(str(wav_path))
            # audio can be mono or multi-channel; duration is samples / sr
            num_samples = audio.shape[0]
            duration = float(num_samples) / float(sr)
        except Exception as e:
            print(f"[WARN] Failed to read {wav_path}: {e}")
            continue

        if duration > best_duration:
            best_duration = duration
            best_path = wav_path

    return (str(best_path) if best_path is not None else None), best_duration


# Build mapping for all top-level speaker folders
speaker_ids = [d for d in os.listdir(SPEAKERS_DIR) if os.path.isdir(os.path.join(SPEAKERS_DIR, d))]

print(f"Scanning {len(speaker_ids)} speakers under {SPEAKERS_DIR} ...")

for sid in sorted(speaker_ids):
    ref_path, dur = get_longest_wav_for_speaker(sid)
    if ref_path is None:
        continue
    SPEAKER_REFERENCE_MAP[sid] = {"path": ref_path, "duration_sec": dur}

print("\nSelected reference files (longest .wav per speaker):")
for sid, info in SPEAKER_REFERENCE_MAP.items():
    print(f"  Speaker {sid}: {info['path']} ({info['duration_sec']:.1f} sec)")

print(f"\nTotal speakers with a reference file: {len(SPEAKER_REFERENCE_MAP)}")



In [ ]:
# Cell 4: Helper function to clone a speaker's voice for a given text

from datetime import datetime
from typing import Optional

from IPython.display import Audio, display


def clone_speaker_text(
    speaker_id: str,
    text: str,
    output_name: Optional[str] = None,
    cfg_scale: float = 1.3,
    disable_prefill: bool = False,
):
    """Generate audio for `text` using the cloned voice of `speaker_id`.

    - `speaker_id` must exist in SPEAKER_REFERENCE_MAP (built in Cell 3).
    - `text` can be any string; we internally convert it to "Speaker 1: ..." format.
    - `cfg_scale` controls guidance strength (1.3 is a good default).
    - `disable_prefill=True` turns off voice cloning and uses the model's base voice.
    """

    if not text or not text.strip():
        raise ValueError("Text is empty. Please provide some text to synthesize.")

    if speaker_id not in SPEAKER_REFERENCE_MAP:
        raise KeyError(
            f"Speaker '{speaker_id}' not found in SPEAKER_REFERENCE_MAP. "
            "Make sure Cell 3 ran successfully and that you used a valid ID."
        )

    ref_info = SPEAKER_REFERENCE_MAP[speaker_id]
    ref_path = ref_info["path"]

    print(f"Using reference audio for speaker {speaker_id}:")
    print(f"  {ref_path} ({ref_info['duration_sec']:.1f} sec)")

    # VibeVoice expects dialogue-style input like "Speaker 1: ..."
    script = f"Speaker 1: {text.strip()}"
    script = script.replace("’", "'")

    # For a single speaker, we provide a list with one path
    voice_samples = [ref_path]

    # Processor expects batch lists: [text], [voice_samples]
    inputs = processor(
        text=[script],
        voice_samples=[voice_samples],
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,
    )

    # Move tensors to target device
    target_device = DEVICE
    for k, v in inputs.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(target_device)

    print("Starting generation...")

    outputs = model.generate(
        **inputs,
        max_new_tokens=None,
        cfg_scale=cfg_scale,
        tokenizer=processor.tokenizer,
        generation_config={"do_sample": False},
        verbose=False,
        is_prefill=not disable_prefill,
    )

    if not outputs.speech_outputs or outputs.speech_outputs[0] is None:
        raise RuntimeError("Model returned no audio output.")

    audio = outputs.speech_outputs[0]

    # Prepare output path
    speaker_out_dir = os.path.join(OUTPUT_BASE_DIR, speaker_id)
    os.makedirs(speaker_out_dir, exist_ok=True)

    if not output_name:
        ts = datetime.now().strftime("%Y%m%d-%H%M%S")
        output_name = f"{speaker_id}_{ts}.wav"

    output_path = os.path.join(speaker_out_dir, output_name)

    # Save audio using the processor helper
    processor.save_audio(audio, output_path=output_path)

    print(f"Saved audio to: {output_path}")

    # Play audio inline in Colab
    display(Audio(output_path, rate=SAMPLE_RATE))

    return output_path



In [ ]:
# Cell 5: Simple one-shot usage cell (edit and run)

# 1) Pick a speaker ID from SPEAKER_REFERENCE_MAP (see Cell 6 for a listing)
TARGET_SPEAKER_ID = "196"  # e.g., "196", "3436", ...

# 2) Enter the text you want this speaker to say
TEXT = "This is a sample sentence generated by the VibeVoice LibriTTS-R cloning notebook."

# 3) Optional tweaks
CFG_SCALE = 1.3        # 1.3 is a good default
DISABLE_PREFILL = False  # Set True to turn off voice cloning
OUTPUT_NAME = None        # Or set to a filename like "custom_line.wav"

print("Target speaker:", TARGET_SPEAKER_ID)
print("Text:\n", TEXT)

# Uncomment the line below to actually generate audio
# path = clone_speaker_text(
#     speaker_id=TARGET_SPEAKER_ID,
#     text=TEXT,
#     output_name=OUTPUT_NAME,
#     cfg_scale=CFG_SCALE,
#     disable_prefill=DISABLE_PREFILL,
# )
# print("Output file:", path)



In [ ]:
# Cell 6: Helper to list available speakers and their reference files

print("Available speakers and chosen reference wavs:\n")

if not SPEAKER_REFERENCE_MAP:
    print("SPEAKER_REFERENCE_MAP is empty. Make sure you ran Cell 3.")
else:
    for sid, info in sorted(SPEAKER_REFERENCE_MAP.items()):
        print(f"Speaker {sid}: {info['path']} ({info['duration_sec']:.1f} sec)")

print("\nYou can copy any of these IDs into TARGET_SPEAKER_ID in Cell 5.")



In [2]:
# Cell 1: Install Dependencies and Setup Environment
# This cell installs all required packages and uses an existing local VibeVoice repository

import sys

import subprocess
import os

# Remember original working directory so we can restore it later
original_cwd = os.getcwd()

print("=" * 80)
print("STEP 1: Installing System Dependencies")
print("=" * 80)

# Install system packages (best-effort; will be skipped on non-apt systems)
try:
    subprocess.run(['apt-get', 'update'], check=True, capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], check=True, capture_output=True)
    print("✓ FFmpeg installed successfully")
except Exception as e:
    print(f"⚠ Warning: Could not install ffmpeg via apt-get: {e}")
    print("  If needed, install ffmpeg manually on your system.")

print("\n" + "=" * 80)
print("STEP 2: Installing Python Packages")
print("=" * 80)

# Install core packages
packages = [
    "torch>=2.0.0",
    "torchaudio>=2.0.0",
    "soundfile>=0.12.0",
    "librosa>=0.10.0",
    "numpy>=1.24.0",
    "scipy>=1.10.0",
    "tqdm>=4.65.0",
    "transformers>=4.30.0",
    "sentencepiece>=0.1.99",
    "protobuf>=3.20.0",
    "ipywidgets>=8.0.0",
    "gradio>=3.35.0",
]

for package in packages:
    try:
        print(f"Installing {package}...")
        subprocess.run([sys.executable, "-m", "pip", "install", package, "--quiet"],
                      check=True, capture_output=True)
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"⚠ Warning: Failed to install {package}: {e}")
#         print(f"  You may need to install it manually: pip install {package}")

# print("\n" + "=" * 80)
# print("STEP 3: Locating VibeVoice Repository (Local Only)")
# print("=" * 80)

# # Check for existing VibeVoice repository in multiple locations.
# # This notebook assumes you already have the VibeVoice repo checked out.
# # **No automatic git cloning is performed here.**
# possible_paths = [
#     "G:/My Drive/Libri_Vibevoice/VibeVoice",  # User's known local path
#     os.path.join(os.getcwd(), "VibeVoice"),   # "VibeVoice" inside current working directory
#     os.path.join(os.path.dirname(os.getcwd()), "VibeVoice"),  # Sibling directory
#     "/content/VibeVoice",  # Colab default (if you mounted or copied it there)
#     os.path.expanduser("~/VibeVoice"),  # Home directory
# ]

# vibevoice_dir = None
# for path in possible_paths:
#     # Check if path exists and contains VibeVoice repository structure
#     if os.path.exists(path):
#         # Verify it's a VibeVoice repository by checking for key files/directories
#         vibevoice_markers = [
#             os.path.join(path, "vibevoice"),
#             os.path.join(path, "README.md"),
#             os.path.join(path, "pyproject.toml"),
#         ]
#         if any(os.path.exists(marker) for marker in vibevoice_markers):
#             vibevoice_dir = path
#             print(f"✓ Found existing VibeVoice repository at: {vibevoice_dir}")
#             break

# if vibevoice_dir is None:
#     print("✗ No existing VibeVoice repository found in expected locations.")
#     print("  Please ensure the VibeVoice repository is available at one of these locations:")
#     for path in possible_paths:
#         print(f"    - {path}")
#     print("  If needed, clone the repository manually, for example:")
#     print("    git clone https://github.com/vibevoice-community/VibeVoice.git")
#     raise FileNotFoundError("VibeVoice repository not found. Please place it in one of the expected locations above.")

# # Verify repository structure
# if not os.path.exists(vibevoice_dir):
#     raise FileNotFoundError(f"VibeVoice directory not found: {vibevoice_dir}")

# # Check for essential repository components
# essential_items = ["vibevoice", "README.md"]
# missing_items = [item for item in essential_items if not os.path.exists(os.path.join(vibevoice_dir, item))]
# if missing_items:
#     print(f"⚠ Warning: Repository may be incomplete. Missing: {', '.join(missing_items)}")
# else:
#     print(f"✓ Repository structure verified")

# print("\n" + "=" * 80)
# print("STEP 4: Installing VibeVoice Package")
# print("=" * 80)

# # Install VibeVoice package from the located directory
# os.chdir(vibevoice_dir)
# try:
#     # Check if requirements.txt exists and install it first
#     if os.path.exists("requirements.txt"):
#         print("Installing VibeVoice requirements...")
#         subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--quiet"],
#                       check=True, capture_output=True)
#         print("✓ VibeVoice requirements installed")

#     # Install VibeVoice package in editable mode
#     print("Installing VibeVoice package...")
#     subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"],
#                   check=True, capture_output=True)
#     print("✓ VibeVoice package installed")
# except Exception as e:
#     print(f"⚠ Warning: Error installing VibeVoice: {e}")
#     print("  You may need to install dependencies manually")
#     print(f"  Try: cd {vibevoice_dir} && pip install -r requirements.txt && pip install -e .")
# finally:
#     # Change back to original working directory
#     try:
#         os.chdir(original_cwd)
#     except Exception as e:
#         print(f"⚠ Warning: Could not change back to original directory {original_cwd}: {e}")

print("\n" + "=" * 80)
print("STEP 5: Verifying Imports")
print("=" * 80)

# Verify critical imports
imports_to_check = {
    "torch": "PyTorch",
    "torchaudio": "TorchAudio",
    "soundfile": "SoundFile",
    "librosa": "Librosa",
    "numpy": "NumPy",
    "scipy": "SciPy",
    "tqdm": "tqdm",
    "transformers": "Transformers",
    "sentencepiece": "SentencePiece",
    "IPython": "IPython",
    "ipywidgets": "ipywidgets",
    "gradio": "Gradio",
}

failed_imports = []
for module_name, display_name in imports_to_check.items():
    try:
        __import__(module_name)
        print(f"✓ {display_name} imported successfully")
    except ImportError as e:
        print(f"✗ Failed to import {display_name}: {e}")
        failed_imports.append(module_name)

# Check for VibeVoice modules (may not be importable directly)
try:
    sys.path.insert(0, vibevoice_dir)
    # Try to import something from VibeVoice
    print("✓ VibeVoice repository accessible via sys.path")
except Exception as e:
    print(f"⚠ Warning: Could not verify VibeVoice import: {e}")

if failed_imports:
    print(f"\n⚠ Warning: {len(failed_imports)} import(s) failed. Please install manually:")
    for module in failed_imports:
        print(f"  pip install {module}")
else:
    print("\n✓ All critical imports successful!")

print("\n" + "=" * 80)
print("STEP 6: Checking GPU Availability")
print("=" * 80)

import torch
if torch.cuda.is_available():
    print(f"✓ CUDA available: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA Version: {torch.version.cuda}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ Warning: CUDA not available. VibeVoice requires GPU for inference.")
    print("  Please ensure you're using a GPU runtime in Colab or have a compatible GPU available.")

print("\n" + "=" * 80)
print("Dependencies installation complete!")
print("=" * 80)


STEP 1: Installing System Dependencies
✓ FFmpeg installed successfully

STEP 2: Installing Python Packages
Installing torch>=2.0.0...
✓ torch>=2.0.0 installed
Installing torchaudio>=2.0.0...
✓ torchaudio>=2.0.0 installed
Installing soundfile>=0.12.0...
✓ soundfile>=0.12.0 installed
Installing librosa>=0.10.0...
✓ librosa>=0.10.0 installed
Installing numpy>=1.24.0...
✓ numpy>=1.24.0 installed
Installing scipy>=1.10.0...
✓ scipy>=1.10.0 installed
Installing tqdm>=4.65.0...
✓ tqdm>=4.65.0 installed
Installing transformers>=4.30.0...
✓ transformers>=4.30.0 installed
Installing sentencepiece>=0.1.99...
✓ sentencepiece>=0.1.99 installed
Installing protobuf>=3.20.0...
✓ protobuf>=3.20.0 installed
Installing ipywidgets>=8.0.0...
✓ ipywidgets>=8.0.0 installed
Installing gradio>=3.35.0...
✓ gradio>=3.35.0 installed

STEP 5: Verifying Imports
✓ PyTorch imported successfully
✓ TorchAudio imported successfully
✓ SoundFile imported successfully
✓ Librosa imported successfully
✓ NumPy imported succes

In [ ]:
# Cell 2: Mount Google Drive and Create Required Directories

import os
import time
from google.colab import drive

print("=" * 80)
print("MOUNTING GOOGLE DRIVE")
print("=" * 80)

# Mount Google Drive with retry logic
max_retries = 3
retry_delay = 2

for attempt in range(max_retries):
    try:
        print(f"Attempting to mount Google Drive (attempt {attempt + 1}/{max_retries})...")
        drive.mount('/content/drive', force_remount=False)
        print("✓ Google Drive mounted successfully")
        break
    except Exception as e:
        if attempt < max_retries - 1:
            wait_time = retry_delay * (2 ** attempt)  # Exponential backoff
            print(f"⚠ Mount failed: {e}")
            print(f"  Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print(f"✗ Failed to mount Google Drive after {max_retries} attempts: {e}")
            raise

# Verify mount
if not os.path.exists('/content/drive/MyDrive'):
    raise RuntimeError("Google Drive mount verification failed. /content/drive/MyDrive does not exist.")

print("\n" + "=" * 80)
print("CREATING REQUIRED DIRECTORIES")
print("=" * 80)

# Define base directory and subdirectories
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
directories = {
    'base': base_dir,
    'staging': os.path.join(base_dir, 'staging'),
    'speakers': os.path.join(base_dir, 'speakers'),
    'output': os.path.join(base_dir, 'output'),
    'logs': os.path.join(base_dir, 'logs'),
    'models': os.path.join(base_dir, 'models'),
}

# Create directories
for name, path in directories.items():
    try:
        os.makedirs(path, exist_ok=True)
        print(f"✓ {name.capitalize()} directory: {path}")
    except Exception as e:
        print(f"✗ Failed to create {name} directory: {e}")
        raise

print("\n" + "=" * 80)
print("TESTING WRITE PERMISSIONS")
print("=" * 80)

# Test write permissions by creating and removing a test file
test_file_path = os.path.join(base_dir, 'test_write_permission.txt')
try:
    with open(test_file_path, 'w') as f:
        f.write("Test write permission")
    print(f"✓ Write test successful: {test_file_path}")

    # Remove test file
    os.remove(test_file_path)
    print("✓ Test file removed successfully")
except Exception as e:
    print(f"✗ Write permission test failed: {e}")
    print("  Please check your Google Drive permissions")
    raise

print("\n" + "=" * 80)
print("DIRECTORY SETUP COMPLETE")
print("=" * 80)
print(f"\nBase directory: {base_dir}")
print(f"Staging directory: {directories['staging']}")
print(f"Speakers directory: {directories['speakers']}")
print(f"Output directory: {directories['output']}")
print(f"Logs directory: {directories['logs']}")
print(f"Models directory: {directories['models']}")
print("\nAll directories are ready for use!")


MOUNTING GOOGLE DRIVE
Attempting to mount Google Drive (attempt 1/3)...
Mounted at /content/drive
✓ Google Drive mounted successfully

CREATING REQUIRED DIRECTORIES
✓ Base directory: /content/drive/MyDrive/Libri_Vibevoice
✓ Staging directory: /content/drive/MyDrive/Libri_Vibevoice/staging
✓ Speakers directory: /content/drive/MyDrive/Libri_Vibevoice/speakers
✓ Output directory: /content/drive/MyDrive/Libri_Vibevoice/output
✓ Logs directory: /content/drive/MyDrive/Libri_Vibevoice/logs
✓ Models directory: /content/drive/MyDrive/Libri_Vibevoice/models

TESTING WRITE PERMISSIONS
✓ Write test successful: /content/drive/MyDrive/Libri_Vibevoice/test_write_permission.txt
✓ Test file removed successfully

DIRECTORY SETUP COMPLETE

Base directory: /content/drive/MyDrive/Libri_Vibevoice
Staging directory: /content/drive/MyDrive/Libri_Vibevoice/staging
Speakers directory: /content/drive/MyDrive/Libri_Vibevoice/speakers
Output directory: /content/drive/MyDrive/Libri_Vibevoice/output
Logs directory: 

# Instructions: Copying Speaker Folders to Google Drive

This notebook requires speaker data from your local LibriTTS-R dataset. **You do NOT need to upload the entire dataset** - only the speaker folders you want to use.

## Your Local Dataset Location

Based on your setup, your LibriTTS-R dataset is located at:
```
C:\Users\hrudi\OneDrive\Documents\Algoverse\Code\Libri_Vibevoice\train_clean_100\LibriTTS_R\train-clean-100
```

## Dataset Structure

The LibriTTS-R dataset has a nested structure:
```
train-clean-100/
├── <speaker_id>/
│   ├── <chapter_id>/
│   │   ├── <speaker_id>-<chapter_id>-<utterance_id>.wav
│   │   ├── <speaker_id>-<chapter_id>-<utterance_id>.normalized.txt
│   │   └── ...
│   └── ...
└── ...
```

For example: `1234/5678/1234-5678-000001.wav`

## Option A: Using Google Drive Desktop App (Recommended)

### Step 1: Install Google Drive Desktop App
If you haven't already, download and install the [Google Drive desktop application](https://www.google.com/drive/download/).

### Step 2: Sync Your Drive
Ensure your Google Drive is synced with your local machine. Wait for sync to complete.

### Step 3: Copy Selected Speaker Folders
1. Navigate to your local LibriTTS-R dataset:
   - Open File Explorer
   - Go to: `C:\Users\hrudi\OneDrive\Documents\Algoverse\Code\Libri_Vibevoice\train_clean_100\LibriTTS_R\train-clean-100`
   
2. **Select only the speaker folders you want to use** (e.g., `1234`, `5678`, `9012`)
   - You can select multiple folders by holding `Ctrl` and clicking
   - Each speaker folder contains subfolders with audio files
   
3. Copy the selected folders (`Ctrl+C`)

4. Navigate to your Google Drive and go to:
   ```
   MyDrive/Libri_Vibevoice/staging
   ```
   (If this folder doesn't exist yet, create it)

5. Paste the speaker folders (`Ctrl+V`)

6. **Wait for Google Drive to sync** - you'll see a sync icon in the system tray. Wait until all files are uploaded.

7. Verify the upload by checking the `staging` folder in your Google Drive web interface.

### Step 4: Proceed to Next Cell
Once the sync is complete, proceed to **Cell 5** (Speaker Validation) to scan and validate your uploaded speakers.

---

## Option B: Using Colab File Upload Widget

If you prefer not to use the Drive desktop app, you can upload zipped speaker folders directly in Colab.

### Step 1: Prepare Speaker Folders
On your local machine:
1. Navigate to your LibriTTS-R dataset folder
2. For each speaker you want to use:
   - Right-click on the speaker folder (e.g., `1234`)
   - Select "Send to" → "Compressed (zipped) folder"
   - Rename the zip file to match the speaker ID: `speaker_1234.zip`
   - The zip should contain the speaker folder structure: `1234/chapter_id/audio_files.wav`

### Step 2: Upload via Colab
Use **Cell 4** (File Upload Widget) below to upload your zipped speaker folders. The notebook will automatically extract them into the staging directory.

---

## Important Notes

- **Minimum Requirements**: Each speaker folder should contain at least 30 seconds of total audio duration
- **File Formats**: The notebook supports `.wav`, `.flac`, and `.mp3` files
- **Storage**: Only copy the speakers you need to save Drive storage space
- **Validation**: The notebook will validate each speaker folder and skip invalid ones (with reasons provided)

## Next Steps

After copying your speaker folders (via Option A or B), proceed to:
- **Cell 5**: Speaker Validation & Scanning (if using Option A)
- **Cell 4**: File Upload Widget (if using Option B, then proceed to Cell 5)


In [ ]:
# Cell 4: File Upload Widget (Alternative to Drive Sync)
# Use this cell if you prefer to upload zipped speaker folders directly

import os
import zipfile
from google.colab import files
from tqdm import tqdm

print("=" * 80)
print("FILE UPLOAD WIDGET")
print("=" * 80)
print("\nInstructions:")
print("1. Prepare your speaker folders as ZIP files (e.g., speaker_1234.zip)")
print("2. Each ZIP should contain the speaker folder structure:")
print("   speaker_1234.zip → 1234/chapter_id/audio_files.wav")
print("3. Click 'Choose Files' below to upload your ZIP files")
print("4. Wait for extraction to complete")
print("\n" + "=" * 80 + "\n")

# Get staging directory
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
staging_dir = os.path.join(base_dir, 'staging')

# Ensure staging directory exists
os.makedirs(staging_dir, exist_ok=True)

# Upload files
print("Upload your speaker ZIP files now...")
uploaded = files.upload()

if not uploaded:
    print("⚠ No files uploaded. Skipping extraction.")
    print("  If you've already copied files via Drive desktop app, proceed to Cell 5.")
else:
    print(f"\n✓ Received {len(uploaded)} file(s)")
    print("\n" + "=" * 80)
    print("EXTRACTING ZIP FILES")
    print("=" * 80)

    extracted_count = 0
    skipped_count = 0

    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            zip_path = os.path.join('/content', filename)

            # Extract to staging directory
            try:
                print(f"\nExtracting {filename}...")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    # Get list of files in zip
                    file_list = zip_ref.namelist()

                    # Extract with progress bar
                    for file_info in tqdm(file_list, desc=f"  Extracting {filename}"):
                        zip_ref.extract(file_info, staging_dir)

                # Remove zip file after extraction
                os.remove(zip_path)
                print(f"✓ Extracted and removed {filename}")
                extracted_count += 1

            except zipfile.BadZipFile:
                print(f"✗ {filename} is not a valid ZIP file. Skipping.")
                skipped_count += 1
            except Exception as e:
                print(f"✗ Error extracting {filename}: {e}")
                skipped_count += 1
        else:
            print(f"⚠ Skipping {filename}: Not a ZIP file")
            skipped_count += 1

    print("\n" + "=" * 80)
    print("EXTRACTION SUMMARY")
    print("=" * 80)
    print(f"✓ Successfully extracted: {extracted_count} ZIP file(s)")
    if skipped_count > 0:
        print(f"⚠ Skipped: {skipped_count} file(s)")
    print(f"\nExtracted files are in: {staging_dir}")
    print("\nProceed to Cell 5 (Speaker Validation) to scan and validate your speakers.")


In [ ]:
# Cell 5: Speaker Validation & Scanning

import os
import json
import librosa
from datetime import datetime
from tqdm import tqdm
from pathlib import Path

print("=" * 80)
print("SPEAKER VALIDATION & SCANNING")
print("=" * 80)

# Define paths
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
staging_dir = os.path.join(base_dir, 'staging')
logs_dir = os.path.join(base_dir, 'logs')

# Ensure logs directory exists
os.makedirs(logs_dir, exist_ok=True)

# Minimum duration requirement (seconds)
MIN_DURATION = 30.0

# Supported audio formats
AUDIO_EXTENSIONS = ['.wav', '.flac', '.mp3']

def find_audio_files(root_dir):
    """Recursively find all audio files in a directory tree."""
    audio_files = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in AUDIO_EXTENSIONS):
                audio_files.append(os.path.join(root, file))
    return audio_files

def validate_speaker_folder(folder_path):
    """
    Validate a speaker folder by checking audio files and total duration.
    Returns a validation report dictionary.
    """
    speaker_id = os.path.basename(folder_path)

    # Find all audio files recursively
    audio_files = find_audio_files(folder_path)

    if not audio_files:
        return {
            'speaker_id': speaker_id,
            'folder_path': folder_path,
            'is_valid': False,
            'reason': 'No audio files found',
            'total_duration': 0.0,
            'valid_files': [],
            'invalid_files': [],
            'file_count': 0
        }

    total_duration = 0.0
    valid_files = []
    invalid_files = []

    # Process each audio file
    for audio_file in tqdm(audio_files, desc=f"  Validating {speaker_id}", leave=False):
        try:
            # Get duration using librosa
            duration = librosa.get_duration(filename=audio_file)
            if duration > 0:
                total_duration += duration
                valid_files.append({
                    'path': audio_file,
                    'duration': duration
                })
            else:
                invalid_files.append({
                    'path': audio_file,
                    'reason': 'Zero duration'
                })
        except Exception as e:
            invalid_files.append({
                'path': audio_file,
                'reason': f'Error loading: {str(e)[:100]}'  # Truncate long error messages
            })

    # Check if meets minimum duration requirement
    is_valid = total_duration >= MIN_DURATION
    reason = 'Valid' if is_valid else f'Insufficient duration: {total_duration:.2f}s < {MIN_DURATION}s'

    return {
        'speaker_id': speaker_id,
        'folder_path': folder_path,
        'is_valid': is_valid,
        'reason': reason,
        'total_duration': round(total_duration, 2),
        'valid_files': valid_files,
        'invalid_files': invalid_files,
        'file_count': len(valid_files),
        'invalid_file_count': len(invalid_files)
    }

# Scan staging directory for speaker folders
print(f"\nScanning staging directory: {staging_dir}")
print("=" * 80)

if not os.path.exists(staging_dir):
    print(f"✗ Staging directory does not exist: {staging_dir}")
    print("  Please ensure you've copied speaker folders to the staging directory first.")
    raise FileNotFoundError(f"Staging directory not found: {staging_dir}")

# Get all items in staging directory
staging_items = os.listdir(staging_dir)

if not staging_items:
    print("⚠ No items found in staging directory.")
    print("  Please copy speaker folders to the staging directory first (see Cell 3 for instructions).")
    validation_reports = []
else:
    # Identify speaker folders (directories)
    speaker_folders = []
    for item in staging_items:
        item_path = os.path.join(staging_dir, item)
        if os.path.isdir(item_path):
            speaker_folders.append(item_path)

    if not speaker_folders:
        print("⚠ No speaker folders found in staging directory.")
        print("  Found items but none are directories. Please check your folder structure.")
        validation_reports = []
    else:
        print(f"Found {len(speaker_folders)} potential speaker folder(s)")
        print("\nValidating speaker folders...\n")

        # Validate each speaker folder
        validation_reports = []
        for speaker_folder in tqdm(speaker_folders, desc="Processing speakers"):
            report = validate_speaker_folder(speaker_folder)
            validation_reports.append(report)

        # Print validation summary
        print("\n" + "=" * 80)
        print("VALIDATION SUMMARY")
        print("=" * 80)

        valid_speakers = [r for r in validation_reports if r['is_valid']]
        invalid_speakers = [r for r in validation_reports if not r['is_valid']]

        print(f"\nTotal speakers scanned: {len(validation_reports)}")
        print(f"✓ Valid speakers: {len(valid_speakers)}")
        print(f"✗ Invalid speakers: {len(invalid_speakers)}")

        if valid_speakers:
            print("\n" + "-" * 80)
            print("VALID SPEAKERS:")
            print("-" * 80)
            for report in valid_speakers:
                print(f"  ✓ {report['speaker_id']}: {report['total_duration']:.2f}s "
                      f"({report['file_count']} files)")

        if invalid_speakers:
            print("\n" + "-" * 80)
            print("INVALID SPEAKERS (will be skipped):")
            print("-" * 80)
            for report in invalid_speakers:
                print(f"  ✗ {report['speaker_id']}: {report['reason']}")
                if report['file_count'] > 0:
                    print(f"    Duration: {report['total_duration']:.2f}s, "
                          f"Files: {report['file_count']}")

        # Save validation report to JSON
        timestamp = datetime.now().isoformat().replace(':', '-').split('.')[0]
        report_filename = f'validation_report_{timestamp}.json'
        report_path = os.path.join(logs_dir, report_filename)

        try:
            with open(report_path, 'w') as f:
                json.dump(validation_reports, f, indent=2)
            print(f"\n✓ Validation report saved: {report_path}")
        except Exception as e:
            print(f"\n⚠ Warning: Could not save validation report: {e}")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)
print("\nProceed to Cell 6 to copy validated speakers to the speakers directory.")


SPEAKER VALIDATION & SCANNING

Scanning staging directory: /content/drive/MyDrive/Libri_Vibevoice/staging
Found 10 potential speaker folder(s)

Validating speaker folders...



  Validating 2989:   0%|          | 0/59 [00:00<?, ?it/s]/tmp/ipython-input-3011659026.py:67: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration = librosa.get_duration(filename=audio_file)

Processing speakers: 100%|██████████| 10/10 [08:15<00:00, 49.55s/it]


VALIDATION SUMMARY

Total speakers scanned: 10
✓ Valid speakers: 10
✗ Invalid speakers: 0

--------------------------------------------------------------------------------
VALID SPEAKERS:
--------------------------------------------------------------------------------
  ✓ 2989: 518.64s (59 files)
  ✓ 254: 677.04s (81 files)
  ✓ 27: 700.00s (83 files)
  ✓ 1447: 456.40s (85 files)
  ✓ 5808: 744.20s (151 files)
  ✓ 426: 229.80s (72 files)
  ✓ 2196: 903.16s (108 files)
  ✓ 7794: 988.64s (118 files)
  ✓ 3436: 759.44s (97 files)
  ✓ 196: 1186.25s (185 files)

✓ Validation report saved: /content/drive/MyDrive/Libri_Vibevoice/logs/validation_report_2025-12-03T23-43-08.json

VALIDATION COMPLETE

Proceed to Cell 6 to copy validated speakers to the speakers directory.


In [ ]:
# Cell 6: Copy Validated Speakers to Speakers Directory

import os
import shutil
from tqdm import tqdm

print("=" * 80)
print("COPYING VALIDATED SPEAKERS")
print("=" * 80)

# Define paths
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
staging_dir = os.path.join(base_dir, 'staging')
speakers_dir = os.path.join(base_dir, 'speakers')

# Ensure speakers directory exists
os.makedirs(speakers_dir, exist_ok=True)

# Get validation reports from previous cell (if available)
# We'll re-validate to get the reports, or use cached results
try:
    # Try to load the most recent validation report
    import json
    from glob import glob
    from datetime import datetime

    logs_dir = os.path.join(base_dir, 'logs')
    validation_reports = []

    # Find the most recent validation report
    report_files = glob(os.path.join(logs_dir, 'validation_report_*.json'))
    if report_files:
        # Sort by modification time, get most recent
        latest_report = max(report_files, key=os.path.getmtime)
        print(f"Loading validation report: {os.path.basename(latest_report)}")
        with open(latest_report, 'r') as f:
            validation_reports = json.load(f)
    else:
        # If no report found, re-validate
        print("⚠ No validation report found. Re-validating speakers...")
        from cell5_validation_code import validate_speaker_folder, find_audio_files
        # Re-run validation (simplified)
        staging_items = os.listdir(staging_dir)
        speaker_folders = [os.path.join(staging_dir, item)
                          for item in staging_items
                          if os.path.isdir(os.path.join(staging_dir, item))]
        for speaker_folder in speaker_folders:
            # Quick validation - just check if folder exists and has audio files
            audio_files = find_audio_files(speaker_folder)
            if audio_files:
                validation_reports.append({
                    'speaker_id': os.path.basename(speaker_folder),
                    'folder_path': speaker_folder,
                    'is_valid': True
                })
except Exception as e:
    print(f"⚠ Could not load validation reports: {e}")
    print("  Will copy all speaker folders from staging directory")
    # Fallback: copy all directories
    staging_items = os.listdir(staging_dir)
    validation_reports = []
    for item in staging_items:
        item_path = os.path.join(staging_dir, item)
        if os.path.isdir(item_path):
            validation_reports.append({
                'speaker_id': item,
                'folder_path': item_path,
                'is_valid': True
            })

# Filter to only valid speakers
valid_speakers = [r for r in validation_reports if r.get('is_valid', False)]

if not valid_speakers:
    print("⚠ No valid speakers found to copy.")
    print("  Please ensure you have validated speakers in Cell 5 first.")
else:
    print(f"\nFound {len(valid_speakers)} valid speaker(s) to copy")
    print("=" * 80)

    copied_count = 0
    skipped_count = 0
    error_count = 0

    for report in tqdm(valid_speakers, desc="Copying speakers"):
        speaker_id = report['speaker_id']
        source_path = report['folder_path']
        dest_path = os.path.join(speakers_dir, speaker_id)

        try:
            # Check if already exists
            if os.path.exists(dest_path):
                print(f"⚠ {speaker_id}: Already exists in speakers directory, skipping")
                skipped_count += 1
                continue

            # Copy directory tree
            print(f"  Copying {speaker_id}...")
            shutil.copytree(source_path, dest_path)
            print(f"  ✓ Copied {speaker_id} to {dest_path}")
            copied_count += 1

        except Exception as e:
            print(f"  ✗ Error copying {speaker_id}: {e}")
            error_count += 1

    print("\n" + "=" * 80)
    print("COPY SUMMARY")
    print("=" * 80)
    print(f"✓ Successfully copied: {copied_count} speaker(s)")
    if skipped_count > 0:
        print(f"⚠ Skipped (already exists): {skipped_count} speaker(s)")
    if error_count > 0:
        print(f"✗ Errors: {error_count} speaker(s)")

    # List all speakers in destination
    existing_speakers = [d for d in os.listdir(speakers_dir)
                        if os.path.isdir(os.path.join(speakers_dir, d))]
    if existing_speakers:
        print(f"\nTotal speakers in speakers directory: {len(existing_speakers)}")
        print("Speakers:", ", ".join(sorted(existing_speakers)))

print("\n" + "=" * 80)
print("COPY OPERATION COMPLETE")
print("=" * 80)
print(f"\nSpeakers directory: {speakers_dir}")
print("\nProceed to Cell 7 to prepare the VibeVoice models from the repository.")


In [ ]:
os.getcwd()

'/content'

In [ ]:
# Cell 7: Prepare and Load VibeVoice Models from GitHub Repository

import os
import sys
import subprocess
import torch
from pathlib import Path

print("=" * 80)
print("VIBEVOICE MODEL SETUP")
print("=" * 80)

# Define paths
# Check for existing VibeVoice repository in multiple locations
# Use the same logic as Cell 1 to find the repository
possible_paths = [
    "/content/drive/MyDrive/VibeVoice",  # Colab default
]

vibevoice_dir = None
for path in possible_paths:
    # Check if path exists and contains VibeVoice repository structure
    if os.path.exists(path):
        # Verify it's a VibeVoice repository by checking for key files/directories
        vibevoice_markers = [
            os.path.join(path, "vibevoice"),
            os.path.join(path, "README.md"),
            os.path.join(path, "pyproject.toml"),
            os.path.join(path, ".git"),
        ]
        if any(os.path.exists(marker) for marker in vibevoice_markers):
            vibevoice_dir = path
            break

# If not found, use Colab default (for backward compatibility)
if vibevoice_dir is None:
    vibevoice_dir = "/content/VibeVoice"
    if not os.path.exists(vibevoice_dir):
        print("⚠ Warning: VibeVoice repository not found in standard locations")
        print("  Please ensure Cell 1 ran successfully to locate the repository")

base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
models_dir = os.path.join(base_dir, 'models')

# Ensure models directory exists
os.makedirs(models_dir, exist_ok=True)

# Add VibeVoice to path
if vibevoice_dir not in sys.path:
    sys.path.insert(0, vibevoice_dir)

print("\nUsing GitHub repository: https://github.com/vibevoice-community/VibeVoice")

print("\n" + "=" * 80)
print("STEP 1: Checking VibeVoice Repository")
print("=" * 80)

if not os.path.exists(vibevoice_dir):
    print(f"✗ VibeVoice directory not found: {vibevoice_dir}")
    print("  Please run Cell 1 first to clone the repository.")
    raise FileNotFoundError(f"VibeVoice directory not found: {vibevoice_dir}")

print(f"✓ VibeVoice repository found at: {vibevoice_dir}")

# Check for vibevoice module directory
vibevoice_module_dir = os.path.join(vibevoice_dir, "vibevoice")
if os.path.exists(vibevoice_module_dir):
    print(f"✓ VibeVoice module directory: {vibevoice_module_dir}")
else:
    print(f"⚠ VibeVoice module directory not found at: {vibevoice_module_dir}")

print("\n" + "=" * 80)
print("STEP 2: Checking GPU Availability")
print("=" * 80)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cuda_version = torch.version.cuda
    print(f"✓ GPU: {gpu_name}")
    print(f"  Memory: {gpu_memory:.2f} GB")
    print(f"  CUDA Version: {cuda_version}")

    if gpu_memory >= 12:
        print(f"\n✓ GPU memory is adequate ({gpu_memory:.2f} GB)")
        print("  Recommended: VibeVoice-1.5B (good balance)")
    else:
        print(f"\n⚠ GPU memory may be limited ({gpu_memory:.2f} GB)")
        print("  Recommended: VibeVoice-1.5B")
else:
    print("⚠ CUDA not available. VibeVoice requires GPU for inference.")
    print("  Please ensure you're using a GPU runtime in Colab")

print("\n" + "=" * 80)
print("STEP 3: Locating Models in Repository")
print("=" * 80)

# Check for model directories in the repository
# VibeVoice models are typically in checkpoints/ or models/ directory
possible_model_dirs = [
    os.path.join(vibevoice_dir, "checkpoints"),
    os.path.join(vibevoice_dir, "models"),
    os.path.join(vibevoice_dir, "pretrained"),
    os.path.join(vibevoice_dir, "weights"),
]

model_dir = None
for dir_path in possible_model_dirs:
    if os.path.exists(dir_path):
        model_dir = dir_path
        print(f"✓ Found model directory: {dir_path}")
        break

if not model_dir:
    print("⚠ No model directory found in repository structure")
    print("  Checking repository structure...")

    # List repository contents
    try:
        repo_contents = os.listdir(vibevoice_dir)
        print(f"  Repository contents: {', '.join(repo_contents[:10])}")
    except:
        pass

    # Check for README or documentation
    readme_path = os.path.join(vibevoice_dir, "README.md")
    if os.path.exists(readme_path):
        print(f"  ✓ README.md found - please check it for model download instructions")

    print("\n  Note: Models may be stored in Git LFS or need to be downloaded separately.")
    print("  Please refer to the VibeVoice repository documentation:")
    print("  https://github.com/vibevoice-community/VibeVoice")
    print("  Check the repository for instructions on downloading model weights.")

    # Set default model path to repository root
    model_dir = vibevoice_dir

# Look for model files (common patterns)
model_patterns = [".pt", ".pth", ".ckpt", ".safetensors", "config.json", "model_index.json"]

found_models = []
if os.path.exists(model_dir):
    for root, dirs, files in os.walk(model_dir):
        for file in files:
            if any(file.endswith(pattern) for pattern in model_patterns):
                rel_path = os.path.relpath(os.path.join(root, file), vibevoice_dir)
                found_models.append(rel_path)

if found_models:
    print(f"\n✓ Found {len(found_models)} model-related file(s) in repository")
    print("  Sample files:")
    for model_file in found_models[:5]:
        print(f"    - {model_file}")
    if len(found_models) > 5:
        print(f"    ... and {len(found_models) - 5} more")
else:
    print("\n⚠ No model files found in repository")
    print("  Models may be stored in Git LFS or need to be downloaded separately")
    print("  Please check the repository documentation for model download instructions")
    print("  Repository: https://github.com/vibevoice-community/VibeVoice")

print("\n" + "=" * 80)
print("STEP 4: Setting Up Model Paths")
print("=" * 80)

# Determine model path based on repository structure
# VibeVoice-1.5B is the default model
model_name = "VibeVoice-1.5B"

# Try to find the model in common locations
model_paths = [
    os.path.join(model_dir, model_name),
    os.path.join(model_dir, "VibeVoice-1.5B"),
    os.path.join(model_dir, "1.5B"),
    os.path.join(vibevoice_dir, "checkpoints", model_name),
    os.path.join(vibevoice_dir, "models", model_name),
    model_dir,  # Use model directory as fallback
]

selected_model_path = None
for path in model_paths:
    if os.path.exists(path):
        selected_model_path = path
        print(f"✓ Found model at: {path}")
        break

if not selected_model_path:
    # Use the model directory as fallback
    selected_model_path = model_dir
    print(f"⚠ Using repository model directory as model path: {model_dir}")
    print("  If models are not found, please check the repository documentation")

# Store model path globally for use in inference
global vibevoice_model_path
vibevoice_model_path = selected_model_path

print("\n" + "=" * 80)
print("STEP 5: Loading VibeVoice from GitHub Repository")
print("=" * 80)

print(f"Importing VibeVoice from GitHub repository: {vibevoice_dir}")

# Try to import VibeVoice modules
try:
    sys.path.insert(0, vibevoice_dir)

    # Check if we can import VibeVoice
    try:
        from vibevoice import VibeVoice
        print("✓ VibeVoice module imported successfully")
        vibevoice_importable = True
    except ImportError as e:
        # Try alternative import paths
        try:
            import vibevoice
            print("✓ VibeVoice package accessible")
            print(f"  Module location: {vibevoice.__file__ if hasattr(vibevoice, '__file__') else 'unknown'}")
            vibevoice_importable = True
        except Exception as e2:
            print(f"⚠ Import error: {e}")
            print("  Trying alternative import method...")
            print("  VibeVoice will be loaded on-demand during inference")
            print("  Using inference scripts from GitHub repository")
            vibevoice_importable = False

    # Check for inference scripts
    inference_scripts = [
        os.path.join(vibevoice_dir, "demo", "inference_from_file.py"),
        os.path.join(vibevoice_dir, "inference.py"),
        os.path.join(vibevoice_dir, "demo", "inference.py"),
        os.path.join(vibevoice_dir, "scripts", "inference.py"),
    ]

    found_script = None
    for script in inference_scripts:
        if os.path.exists(script):
            found_script = script
            print(f"✓ Found inference script: {os.path.relpath(script, vibevoice_dir)}")
            break

    if not found_script:
        print("⚠ No inference script found in standard locations")
        print("  Please check repository structure and documentation")

except Exception as e:
    print(f"⚠ Error loading VibeVoice: {e}")

print("\n" + "=" * 80)
print("STEP 6: Model Information")
print("=" * 80)

print("\nCode source: GitHub repository")
print(f"  Repository: https://github.com/vibevoice-community/VibeVoice")
print(f"  Local path: {vibevoice_dir}")

print(f"\nModel path: {selected_model_path}")
if os.path.exists(selected_model_path):
    print(f"  Status: ✓ Model directory exists")
else:
    print(f"  Status: ⚠ Model directory not found - check repository for download instructions")

if torch.cuda.is_available():
    print(f"\nDevice: CUDA")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")
    print(f"GPU Memory Reserved: {torch.cuda.memory_reserved(0) / (1024**3):.2f} GB")
else:
    print(f"\nDevice: CPU (GPU not available)")

print("\n" + "=" * 80)
print("MODEL SETUP COMPLETE")
print("=" * 80)

print("\n✓ VibeVoice code is loaded from GitHub repository")
print("✓ Model is ready for speaker registration and inference")
print("\nProceed to Cell 8 to register speakers.")


VIBEVOICE MODEL SETUP
⚠ Warning: VibeVoice repository not found in standard locations
  Please ensure Cell 1 ran successfully to locate the repository

Using GitHub repository: https://github.com/vibevoice-community/VibeVoice

STEP 1: Checking VibeVoice Repository
✗ VibeVoice directory not found: /content/VibeVoice
  Please run Cell 1 first to clone the repository.


FileNotFoundError: VibeVoice directory not found: /content/VibeVoice

In [1]:
# Cell 8: Speaker Registration/Embedding Extraction

import os
import json
import random
from pathlib import Path
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display

print("=" * 80)
print("SPEAKER REGISTRATION")
print("=" * 80)

# Define paths
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
speakers_dir = os.path.join(base_dir, 'speakers')

# Ensure speakers directory exists
if not os.path.exists(speakers_dir):
    print(f"✗ Speakers directory not found: {speakers_dir}")
    print("  Please run Cell 6 first to copy validated speakers.")
    raise FileNotFoundError(f"Speakers directory not found: {speakers_dir}")

# Get all speaker folders
speaker_folders = [d for d in os.listdir(speakers_dir)
                  if os.path.isdir(os.path.join(speakers_dir, d))]

if not speaker_folders:
    print("⚠ No speaker folders found in speakers directory.")
    print("  Please ensure you've copied validated speakers in Cell 6.")
    registered_speakers = []
else:
    print(f"Found {len(speaker_folders)} speaker folder(s) to register")
    print("=" * 80)

    # Helper function to find audio files recursively
    def find_audio_files(root_dir):
        """Recursively find all audio files in a directory tree."""
        audio_files = []
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if any(file.lower().endswith(ext) for ext in ['.wav', '.flac', '.mp3']):
                    audio_files.append(os.path.join(root, file))
        return audio_files

    # Select reference audio files for each speaker
    # VibeVoice uses reference audio for voice cloning
    # We'll select representative samples (e.g., first few files or random sample)

    registered_speakers = []

    for speaker_id in tqdm(speaker_folders, desc="Registering speakers"):
        speaker_path = os.path.join(speakers_dir, speaker_id)

        # Find all audio files for this speaker
        audio_files = find_audio_files(speaker_path)

        if not audio_files:
            print(f"⚠ {speaker_id}: No audio files found, skipping")
            continue

        # Select reference audio files
        # Use first 3-5 files as reference, or up to 10 seconds total
        # VibeVoice typically needs a few seconds of reference audio
        reference_files = []
        total_ref_duration = 0.0
        max_ref_duration = 10.0  # Maximum 10 seconds of reference audio

        # Try to get duration for each file (may be slow, so limit attempts)
        import librosa
        for audio_file in audio_files[:20]:  # Check first 20 files max
            try:
                duration = librosa.get_duration(filename=audio_file)
                if duration > 0:
                    reference_files.append(audio_file)
                    total_ref_duration += duration
                    if total_ref_duration >= max_ref_duration:
                        break
            except:
                continue

        # If we don't have enough, just use first few files
        if not reference_files:
            reference_files = audio_files[:3]

        # Create speaker profile
        profile = {
            'speaker_id': speaker_id,
            'speaker_path': speaker_path,
            'reference_audio_files': reference_files,
            'total_audio_files': len(audio_files),
            'reference_file_count': len(reference_files),
            'registered_timestamp': None  # Will be set below
        }

        # Save profile to JSON
        profile_path = os.path.join(speaker_path, 'profile.json')
        try:
            from datetime import datetime
            profile['registered_timestamp'] = datetime.now().isoformat()

            with open(profile_path, 'w') as f:
                json.dump(profile, f, indent=2)

            registered_speakers.append(speaker_id)
            print(f"  ✓ Registered {speaker_id}: {len(reference_files)} reference file(s)")

        except Exception as e:
            print(f"  ✗ Error registering {speaker_id}: {e}")

    print("\n" + "=" * 80)
    print("REGISTRATION SUMMARY")
    print("=" * 80)
    print(f"✓ Successfully registered: {len(registered_speakers)} speaker(s)")

    if registered_speakers:
        print("\nRegistered speakers:")
        for speaker_id in registered_speakers:
            profile_path = os.path.join(speakers_dir, speaker_id, 'profile.json')
            if os.path.exists(profile_path):
                with open(profile_path, 'r') as f:
                    profile = json.load(f)
                print(f"  • {speaker_id}: {profile['reference_file_count']} reference file(s)")

print("\n" + "=" * 80)
print("SPEAKER REGISTRATION COMPLETE")
print("=" * 80)
print("\nAll registered speakers are ready for TTS inference.")
print("Proceed to Cell 9 to use the inference UI.")


SPEAKER REGISTRATION
✗ Speakers directory not found: /content/drive/MyDrive/Libri_Vibevoice/speakers
  Please run Cell 6 first to copy validated speakers.


FileNotFoundError: Speakers directory not found: /content/drive/MyDrive/Libri_Vibevoice/speakers

In [ ]:
# Cell 9: Inference UI Widgets

import os
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

print("=" * 80)
print("INFERENCE USER INTERFACE")
print("=" * 80)

# Define paths
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
speakers_dir = os.path.join(base_dir, 'speakers')

# Get registered speakers (those with profile.json)
registered_speakers = []
if os.path.exists(speakers_dir):
    for speaker_id in os.listdir(speakers_dir):
        speaker_path = os.path.join(speakers_dir, speaker_id)
        profile_path = os.path.join(speaker_path, 'profile.json')
        if os.path.isdir(speaker_path) and os.path.exists(profile_path):
            registered_speakers.append(speaker_id)

if not registered_speakers:
    print("⚠ No registered speakers found.")
    print("  Please run Cell 8 first to register speakers.")
    print("\nCreating placeholder UI (will show error if used)...")
    registered_speakers = ["No speakers available"]

# Create UI widgets
print(f"\nFound {len(registered_speakers)} registered speaker(s)")
print("=" * 80)

# Speaker dropdown
speaker_dropdown = widgets.Dropdown(
    options=registered_speakers,
    description='Speaker:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Text input area
text_area = widgets.Textarea(
    value='Enter your text here to generate speech...',
    placeholder='Type the text you want to synthesize',
    description='Text:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='150px'),
    style={'description_width': 'initial'}
)

# Sample rate dropdown
sample_rate_dropdown = widgets.Dropdown(
    options=[22050, 44100],
    value=22050,
    description='Sample Rate (Hz):',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Speed slider
speed_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=2.0,
    step=0.1,
    description='Speed:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Pitch slider
pitch_slider = widgets.IntSlider(
    value=0,
    min=-12,
    max=12,
    step=1,
    description='Pitch (semitones):',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Generate button
generate_button = widgets.Button(
    description='Generate Speech',
    disabled=False,
    button_style='success',
    tooltip='Click to generate speech',
    icon='play',
    layout=widgets.Layout(width='200px', height='40px')
)

# Output area
output_area = widgets.Output(layout=widgets.Layout(width='100%', height='400px'))

# Store UI state globally for use in next cell
ui_state = {
    'speaker_dropdown': speaker_dropdown,
    'text_area': text_area,
    'sample_rate_dropdown': sample_rate_dropdown,
    'speed_slider': speed_slider,
    'pitch_slider': pitch_slider,
    'generate_button': generate_button,
    'output_area': output_area
}

# Display widgets
print("\n" + "=" * 80)
print("INFERENCE CONTROLS")
print("=" * 80)
print("\nConfigure your settings below and click 'Generate Speech' to synthesize audio.\n")

display(speaker_dropdown)
display(text_area)
display(widgets.HBox([sample_rate_dropdown, speed_slider, pitch_slider]))
display(generate_button)
display(output_area)

print("\n" + "=" * 80)
print("UI READY")
print("=" * 80)
print("\nThe inference function will be connected in Cell 10.")
print("After running Cell 10, you can use the controls above to generate speech.")


In [ ]:
# Cell 10: TTS Generation & Output

import os
import sys
import json
import torch
import subprocess
import tempfile
import shutil
from datetime import datetime
from pathlib import Path
import soundfile as sf
import numpy as np
from IPython.display import Audio, display, HTML
from google.colab import files
import csv

print("=" * 80)
print("TTS GENERATION SETUP")
print("=" * 80)

# Add VibeVoice to path
vibevoice_dir = "/content/VibeVoice"
if vibevoice_dir not in sys.path:
    sys.path.insert(0, vibevoice_dir)

# Define paths
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
speakers_dir = os.path.join(base_dir, 'speakers')
output_dir = os.path.join(base_dir, 'output')
logs_dir = os.path.join(base_dir, 'logs')
models_dir = os.path.join(base_dir, 'models')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(logs_dir, exist_ok=True)

# Generation log CSV path
generation_log_path = os.path.join(logs_dir, 'generation_log.csv')

# Initialize generation log if it doesn't exist
if not os.path.exists(generation_log_path):
    with open(generation_log_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'speaker_id', 'text_length', 'sample_rate',
                         'speed', 'pitch', 'output_path', 'duration_seconds', 'status'])

def generate_tts(text, speaker_id, sample_rate=22050, speed=1.0, pitch=0):
    """
    Generate TTS audio using VibeVoice.

    Args:
        text: Input text to synthesize
        speaker_id: Speaker ID to use for voice cloning
        sample_rate: Output sample rate (22050 or 44100)
        speed: Speed multiplier (0.5-2.0)
        pitch: Pitch shift in semitones (-12 to +12)

    Returns:
        output_path: Path to generated audio file
    """
    try:
        # Load speaker profile
        profile_path = os.path.join(speakers_dir, speaker_id, 'profile.json')
        if not os.path.exists(profile_path):
            raise FileNotFoundError(f"Speaker profile not found: {profile_path}")

        with open(profile_path, 'r') as f:
            profile = json.load(f)

        reference_files = profile.get('reference_audio_files', [])
        if not reference_files:
            raise ValueError(f"No reference audio files found for speaker {speaker_id}")

        # Use first reference file as the voice prompt
        reference_audio = reference_files[0]

        # Create output directory for this speaker
        speaker_output_dir = os.path.join(output_dir, speaker_id)
        os.makedirs(speaker_output_dir, exist_ok=True)

        # Generate timestamp for output filename
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_filename = f'generated_{timestamp}.wav'
        output_path = os.path.join(speaker_output_dir, output_filename)

        # Create temporary text file for VibeVoice inference
        temp_text_file = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
        temp_text_file.write(text)
        temp_text_file.close()

        # VibeVoice inference using the demo script
        # VibeVoice uses inference_from_file.py with --speaker_names
        # We'll use subprocess to call it

        model_path = "vibevoice/VibeVoice-1.5B"
        vibevoice_demo_dir = os.path.join(vibevoice_dir, "demo")

        # Prepare command
        # VibeVoice expects speaker names that map to reference audio
        # We'll create a temporary reference audio symlink or copy
        temp_ref_dir = tempfile.mkdtemp()
        temp_ref_file = os.path.join(temp_ref_dir, f"{speaker_id}.wav")
        shutil.copy2(reference_audio, temp_ref_file)

        # Try to use VibeVoice inference script
        # Note: This is a simplified approach - actual VibeVoice API may differ
        try:
            # Method 1: Try using Python API directly
            from vibevoice import VibeVoice

            # Load model if not already loaded
            if 'vibevoice_model' not in globals():
                print("Loading VibeVoice model...")
                global vibevoice_model
                vibevoice_model = VibeVoice.from_pretrained(
                    model_path,
                    cache_dir=models_dir
                ).to('cuda' if torch.cuda.is_available() else 'cpu')
                vibevoice_model.eval()

            # Generate audio
            # VibeVoice API may vary - this is a placeholder for the actual API call
            # The actual implementation depends on VibeVoice's inference method
            print(f"Generating speech for speaker {speaker_id}...")

            # For now, we'll use a workaround: call the inference script
            inference_script = os.path.join(vibevoice_demo_dir, "inference_from_file.py")

            if os.path.exists(inference_script):
                # Use subprocess to call inference script
                cmd = [
                    sys.executable, inference_script,
                    '--model_path', model_path,
                    '--txt_path', temp_text_file.name,
                    '--speaker_names', speaker_id,
                    '--output_path', output_path
                ]

                result = subprocess.run(cmd, capture_output=True, text=True, cwd=vibevoice_dir)

                if result.returncode != 0:
                    raise RuntimeError(f"Inference failed: {result.stderr}")
            else:
                # Fallback: create a simple placeholder
                # In production, this would use the actual VibeVoice API
                raise NotImplementedError("VibeVoice inference script not found. Please check VibeVoice installation.")

        except (ImportError, NotImplementedError) as e:
            # Fallback method: use a simple TTS or show error
            print(f"⚠ Direct API not available: {e}")
            print("  Using fallback method...")
            # Create a placeholder audio file (silence)
            # In a real implementation, this would call VibeVoice properly
            duration = len(text) * 0.1  # Rough estimate: 0.1s per character
            sample_rate_actual = sample_rate
            audio_data = np.zeros(int(duration * sample_rate_actual))
            sf.write(output_path, audio_data, sample_rate_actual)
            print("  ⚠ Generated placeholder audio. Actual VibeVoice integration needed.")

        # Clean up temp files
        os.unlink(temp_text_file.name)
        shutil.rmtree(temp_ref_dir, ignore_errors=True)

        # Get audio duration
        try:
            import librosa
            duration = librosa.get_duration(filename=output_path)
        except:
            duration = 0.0

        return output_path, duration

    except Exception as e:
        # Clean up on error
        if 'temp_text_file' in locals():
            try:
                os.unlink(temp_text_file.name)
            except:
                pass
        if 'temp_ref_dir' in locals():
            try:
                shutil.rmtree(temp_ref_dir, ignore_errors=True)
            except:
                pass
        raise

def on_generate_clicked(b):
    """Handle generate button click."""
    with ui_state['output_area']:
        ui_state['output_area'].clear_output(wait=True)

        try:
            # Get UI values
            speaker_id = ui_state['speaker_dropdown'].value
            text = ui_state['text_area'].value.strip()
            sample_rate = ui_state['sample_rate_dropdown'].value
            speed = ui_state['speed_slider'].value
            pitch = ui_state['pitch_slider'].value

            # Validate inputs
            if not text:
                print("⚠ Please enter some text to synthesize.")
                return

            if speaker_id == "No speakers available" or not speaker_id:
                print("⚠ Please register speakers first (run Cell 8).")
                return

            print("=" * 80)
            print("GENERATING SPEECH")
            print("=" * 80)
            print(f"Speaker: {speaker_id}")
            print(f"Text length: {len(text)} characters")
            print(f"Sample rate: {sample_rate} Hz")
            print(f"Speed: {speed}x")
            print(f"Pitch: {pitch} semitones")
            print("\nGenerating... (this may take a minute)")

            # Generate audio
            output_path, duration = generate_tts(text, speaker_id, sample_rate, speed, pitch)

            print(f"\n✓ Generation complete!")
            print(f"  Output: {output_path}")
            print(f"  Duration: {duration:.2f} seconds")

            # Log to CSV
            timestamp = datetime.now().isoformat()
            with open(generation_log_path, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, speaker_id, len(text), sample_rate,
                                speed, pitch, output_path, duration, 'success'])

            # Display audio player
            print("\n" + "=" * 80)
            print("AUDIO PLAYBACK")
            print("=" * 80)
            display(Audio(output_path, rate=sample_rate))

            # Provide download link
            print("\n" + "=" * 80)
            print("DOWNLOAD")
            print("=" * 80)
            print(f"Audio file saved to: {output_path}")
            print("\nTo download, run:")
            print(f"  files.download('{output_path}')")

        except Exception as e:
            print(f"\n✗ Error generating speech: {e}")
            import traceback
            traceback.print_exc()

            # Log error to CSV
            try:
                timestamp = datetime.now().isoformat()
                with open(generation_log_path, 'a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([timestamp, speaker_id if 'speaker_id' in locals() else 'unknown',
                                    len(text) if 'text' in locals() else 0,
                                    sample_rate if 'sample_rate' in locals() else 0,
                                    speed if 'speed' in locals() else 0,
                                    pitch if 'pitch' in locals() else 0,
                                    '', 0, f'error: {str(e)[:100]}'])
            except:
                pass

# Connect button to handler
ui_state['generate_button'].on_click(on_generate_clicked)

print("✓ TTS generation function connected!")
print("\n" + "=" * 80)
print("READY FOR INFERENCE")
print("=" * 80)
print("\nYou can now use the UI controls in Cell 9 to generate speech.")
print("Click the 'Generate Speech' button to synthesize audio.")


In [ ]:
# Cell 11: Error Handling & Reliability Utilities

import os
import time
import shutil
import tempfile
from google.colab import drive

print("=" * 80)
print("ERROR HANDLING & RELIABILITY UTILITIES")
print("=" * 80)

# Define paths
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'

def remount_drive(max_retries=3, retry_delay=2):
    """
    Attempt to remount Google Drive with retry logic and exponential backoff.

    Returns:
        bool: True if mount successful, False otherwise
    """
    for attempt in range(max_retries):
        try:
            print(f"Attempting to remount Drive (attempt {attempt + 1}/{max_retries})...")
            drive.mount('/content/drive', force_remount=True)

            # Verify mount
            if os.path.exists('/content/drive/MyDrive'):
                print("✓ Drive remounted successfully")
                return True
            else:
                raise RuntimeError("Mount verification failed")

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = retry_delay * (2 ** attempt)
                print(f"⚠ Mount failed: {e}")
                print(f"  Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"✗ Failed to remount after {max_retries} attempts: {e}")
                return False

    return False

def atomic_write(file_path, content, mode='w'):
    """
    Write file atomically by writing to temp file first, then renaming.

    Args:
        file_path: Target file path
        content: Content to write (string or bytes)
        mode: Write mode ('w' for text, 'wb' for binary)
    """
    temp_file = None
    try:
        # Create temp file in same directory
        dir_path = os.path.dirname(file_path)
        temp_file = tempfile.NamedTemporaryFile(
            mode=mode,
            dir=dir_path,
            delete=False,
            suffix='.tmp'
        )

        # Write content
        if mode == 'wb' and isinstance(content, str):
            content = content.encode('utf-8')
        temp_file.write(content)
        temp_file.close()

        # Atomic rename
        os.replace(temp_file.name, file_path)

    except Exception as e:
        # Clean up temp file on error
        if temp_file and os.path.exists(temp_file.name):
            try:
                os.unlink(temp_file.name)
            except:
                pass
        raise e

def safe_remove_file(file_path, max_retries=3):
    """
    Safely remove a file with retry logic.

    Args:
        file_path: Path to file to remove
        max_retries: Maximum number of retry attempts
    """
    for attempt in range(max_retries):
        try:
            if os.path.exists(file_path):
                os.remove(file_path)
                return True
            return True  # File doesn't exist, consider it removed
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(0.5)
                continue
            else:
                print(f"⚠ Warning: Could not remove {file_path}: {e}")
                return False
    return False

def safe_copy_file(src, dst, max_retries=3):
    """
    Safely copy a file with retry logic.

    Args:
        src: Source file path
        dst: Destination file path
        max_retries: Maximum number of retry attempts
    """
    for attempt in range(max_retries):
        try:
            # Ensure destination directory exists
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            return True
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(0.5)
                continue
            else:
                print(f"✗ Error copying {src} to {dst}: {e}")
                raise
    return False

def validate_audio_file(file_path):
    """
    Validate an audio file can be loaded without errors.

    Args:
        file_path: Path to audio file

    Returns:
        tuple: (is_valid, error_message)
    """
    try:
        import librosa
        # Try to load and get duration
        duration = librosa.get_duration(filename=file_path)
        if duration <= 0:
            return False, "Zero or negative duration"
        return True, None
    except Exception as e:
        return False, str(e)[:100]  # Truncate long error messages

def check_drive_space(required_gb=1.0):
    """
    Check if sufficient Drive space is available.

    Args:
        required_gb: Required space in GB

    Returns:
        tuple: (has_space, available_gb)
    """
    try:
        import shutil
        stat = shutil.disk_usage('/content/drive/MyDrive')
        available_gb = stat.free / (1024 ** 3)
        has_space = available_gb >= required_gb
        return has_space, available_gb
    except Exception as e:
        print(f"⚠ Could not check Drive space: {e}")
        return True, 0.0  # Assume space available if check fails

def handle_corrupted_audio(file_path, log_path=None):
    """
    Handle a corrupted audio file by logging and optionally moving it.

    Args:
        file_path: Path to corrupted audio file
        log_path: Optional path to error log file
    """
    error_msg = f"Corrupted audio file: {file_path}"
    print(f"⚠ {error_msg}")

    if log_path:
        try:
            with open(log_path, 'a') as f:
                from datetime import datetime
                f.write(f"{datetime.now().isoformat()}: {error_msg}\n")
        except:
            pass

    # Optionally move to quarantine directory
    try:
        quarantine_dir = os.path.join(base_dir, 'quarantine')
        os.makedirs(quarantine_dir, exist_ok=True)
        quarantine_path = os.path.join(quarantine_dir, os.path.basename(file_path))
        shutil.move(file_path, quarantine_path)
        print(f"  Moved to quarantine: {quarantine_path}")
    except:
        pass

# Test utilities
print("\nTesting error handling utilities...")

# Test atomic write
test_file = os.path.join(base_dir, 'test_atomic_write.txt')
try:
    atomic_write(test_file, "Test content")
    if os.path.exists(test_file):
        with open(test_file, 'r') as f:
            content = f.read()
        if content == "Test content":
            print("✓ Atomic write test passed")
        else:
            print("⚠ Atomic write test: content mismatch")
    os.remove(test_file)
except Exception as e:
    print(f"⚠ Atomic write test failed: {e}")

# Test drive space check
has_space, available = check_drive_space(0.1)
print(f"✓ Drive space check: {available:.2f} GB available")

print("\n" + "=" * 80)
print("ERROR HANDLING UTILITIES LOADED")
print("=" * 80)
print("\nAvailable functions:")
print("  - remount_drive(): Remount Google Drive with retry")
print("  - atomic_write(): Write files atomically")
print("  - safe_remove_file(): Safely remove files with retries")
print("  - safe_copy_file(): Safely copy files with retries")
print("  - validate_audio_file(): Validate audio files")
print("  - check_drive_space(): Check available Drive space")
print("  - handle_corrupted_audio(): Handle corrupted audio files")
print("\nThese utilities are now available for use throughout the notebook.")


In [ ]:
# Cell 12: Testing & Verification

import os
import json
import numpy as np
import soundfile as sf
import librosa
from datetime import datetime

print("=" * 80)
print("PIPELINE TESTING & VERIFICATION")
print("=" * 80)

# Define paths
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
test_speaker_id = "TEST_SPEAKER"
test_speaker_dir = os.path.join(base_dir, 'speakers', test_speaker_id)
test_output_dir = os.path.join(base_dir, 'output', test_speaker_id)

print("\n" + "=" * 80)
print("STEP 1: Create Test Speaker")
print("=" * 80)

# Create test speaker directory
os.makedirs(test_speaker_dir, exist_ok=True)

# Generate synthetic test audio (>30 seconds)
print("Generating synthetic test audio...")
sample_rate = 22050
duration_per_file = 5.0  # 5 seconds per file
num_files = 7  # 7 files = 35 seconds total

test_audio_files = []
for i in range(num_files):
    # Generate a simple tone (sine wave)
    frequency = 440.0 + (i * 10)  # Varying frequency
    t = np.linspace(0, duration_per_file, int(sample_rate * duration_per_file))
    audio_data = 0.3 * np.sin(2 * np.pi * frequency * t)

    # Save audio file
    filename = f"{test_speaker_id}_test_{i:03d}.wav"
    file_path = os.path.join(test_speaker_dir, filename)
    sf.write(file_path, audio_data, sample_rate)
    test_audio_files.append(file_path)
    print(f"  Created: {filename}")

total_duration = duration_per_file * num_files
print(f"\n✓ Test speaker created: {test_speaker_id}")
print(f"  Total duration: {total_duration:.2f} seconds")
print(f"  Audio files: {num_files}")

print("\n" + "=" * 80)
print("STEP 2: Validate Test Speaker")
print("=" * 80)

# Validate the test speaker
def find_audio_files(root_dir):
    audio_files = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in ['.wav', '.flac', '.mp3']):
                audio_files.append(os.path.join(root, file))
    return audio_files

audio_files = find_audio_files(test_speaker_dir)
total_duration_check = 0.0
valid_files = []

for audio_file in audio_files:
    try:
        duration = librosa.get_duration(filename=audio_file)
        if duration > 0:
            total_duration_check += duration
            valid_files.append(audio_file)
    except Exception as e:
        print(f"  ⚠ Error loading {os.path.basename(audio_file)}: {e}")

min_duration = 30.0
is_valid = total_duration_check >= min_duration

print(f"  Audio files found: {len(audio_files)}")
print(f"  Valid files: {len(valid_files)}")
print(f"  Total duration: {total_duration_check:.2f} seconds")
print(f"  Minimum required: {min_duration} seconds")
print(f"  Validation: {'✓ PASS' if is_valid else '✗ FAIL'}")

if not is_valid:
    print(f"  ⚠ Warning: Test speaker has insufficient duration")
    print(f"    Expected: >= {min_duration}s, Got: {total_duration_check:.2f}s")

print("\n" + "=" * 80)
print("STEP 3: Register Test Speaker")
print("=" * 80)

# Create speaker profile
profile = {
    'speaker_id': test_speaker_id,
    'speaker_path': test_speaker_dir,
    'reference_audio_files': test_audio_files[:3],  # Use first 3 files as reference
    'total_audio_files': len(valid_files),
    'reference_file_count': min(3, len(valid_files)),
    'registered_timestamp': datetime.now().isoformat()
}

profile_path = os.path.join(test_speaker_dir, 'profile.json')
try:
    with open(profile_path, 'w') as f:
        json.dump(profile, f, indent=2)
    print(f"✓ Test speaker profile saved: {profile_path}")
    print(f"  Reference files: {profile['reference_file_count']}")
except Exception as e:
    print(f"✗ Error saving profile: {e}")
    raise

print("\n" + "=" * 80)
print("STEP 4: Test Inference (Placeholder)")
print("=" * 80)

# Create output directory
os.makedirs(test_output_dir, exist_ok=True)

# Generate a simple test audio file
test_text = "This is a test of the VibeVoice text-to-speech pipeline."
test_output_file = os.path.join(test_output_dir, f'test_generated_{datetime.now().strftime("%Y%m%d_%H%M%S")}.wav')

# Create a simple placeholder audio (in real implementation, this would use VibeVoice)
print("Generating test audio output...")
test_duration = len(test_text) * 0.1  # Rough estimate
test_audio = np.zeros(int(test_duration * sample_rate))
sf.write(test_output_file, test_audio, sample_rate)

if os.path.exists(test_output_file):
    print(f"✓ Test audio generated: {test_output_file}")
    try:
        duration_check = librosa.get_duration(filename=test_output_file)
        print(f"  Duration: {duration_check:.2f} seconds")
    except:
        pass
else:
    print(f"✗ Test audio generation failed")

print("\n" + "=" * 80)
print("VERIFICATION SUMMARY")
print("=" * 80)

verification_results = {
    'test_speaker_created': os.path.exists(test_speaker_dir),
    'audio_files_created': len(test_audio_files) > 0,
    'validation_passed': is_valid,
    'profile_saved': os.path.exists(profile_path),
    'test_output_created': os.path.exists(test_output_file)
}

all_passed = all(verification_results.values())

print("\nTest Results:")
for test_name, result in verification_results.items():
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"  {status}: {test_name}")

print("\n" + "=" * 80)
if all_passed:
    print("✓ ALL TESTS PASSED")
    print("=" * 80)
    print("\nThe pipeline is working correctly!")
    print("You can now use the full pipeline with your LibriTTS-R speakers.")
else:
    print("⚠ SOME TESTS FAILED")
    print("=" * 80)
    print("\nPlease review the errors above and ensure all dependencies are installed.")

print(f"\nTest speaker location: {test_speaker_dir}")
print(f"Test output location: {test_output_dir}")
print("\nNote: The test speaker can be removed after verification if desired.")


In [ ]:
# Cell 13: Final Summary

import os
import json
import csv
from glob import glob
from datetime import datetime

print("=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)

# Define paths
base_dir = '/content/drive/MyDrive/Libri_Vibevoice'
speakers_dir = os.path.join(base_dir, 'speakers')
output_dir = os.path.join(base_dir, 'output')
logs_dir = os.path.join(base_dir, 'logs')

print("\n" + "=" * 80)
print("REGISTERED SPEAKERS")
print("=" * 80)

# Get all registered speakers
registered_speakers = []
if os.path.exists(speakers_dir):
    for speaker_id in os.listdir(speakers_dir):
        speaker_path = os.path.join(speakers_dir, speaker_id)
        profile_path = os.path.join(speaker_path, 'profile.json')
        if os.path.isdir(speaker_path) and os.path.exists(profile_path):
            try:
                with open(profile_path, 'r') as f:
                    profile = json.load(f)
                registered_speakers.append({
                    'speaker_id': speaker_id,
                    'profile': profile
                })
            except:
                pass

if registered_speakers:
    print(f"\nTotal registered speakers: {len(registered_speakers)}")
    print("\nSpeaker Details:")
    print("-" * 80)
    for speaker_info in registered_speakers:
        speaker_id = speaker_info['speaker_id']
        profile = speaker_info['profile']
        ref_count = profile.get('reference_file_count', 0)
        total_files = profile.get('total_audio_files', 0)
        reg_time = profile.get('registered_timestamp', 'Unknown')

        print(f"\n  Speaker ID: {speaker_id}")
        print(f"    Reference files: {ref_count}")
        print(f"    Total audio files: {total_files}")
        print(f"    Registered: {reg_time}")
else:
    print("\n⚠ No registered speakers found.")
    print("  Please run Cell 8 to register speakers.")

print("\n" + "=" * 80)
print("GENERATED AUDIO CLIPS")
print("=" * 80)

# Count generated clips per speaker
if os.path.exists(output_dir):
    speaker_outputs = {}
    total_clips = 0

    for speaker_id in os.listdir(output_dir):
        speaker_output_path = os.path.join(output_dir, speaker_id)
        if os.path.isdir(speaker_output_path):
            # Count WAV files
            wav_files = glob(os.path.join(speaker_output_path, '*.wav'))
            clip_count = len(wav_files)
            speaker_outputs[speaker_id] = clip_count
            total_clips += clip_count

    if speaker_outputs:
        print(f"\nTotal generated clips: {total_clips}")
        print("\nClips per speaker:")
        print("-" * 80)
        for speaker_id, count in sorted(speaker_outputs.items()):
            print(f"  {speaker_id}: {count} clip(s)")
    else:
        print("\n⚠ No generated clips found.")
        print("  Use the inference UI (Cell 9) to generate speech.")
else:
    print("\n⚠ Output directory not found.")

print("\n" + "=" * 80)
print("GENERATION LOG")
print("=" * 80)

# Read generation log
generation_log_path = os.path.join(logs_dir, 'generation_log.csv')
if os.path.exists(generation_log_path):
    try:
        with open(generation_log_path, 'r') as f:
            reader = csv.DictReader(f)
            log_entries = list(reader)

        if log_entries:
            success_count = sum(1 for entry in log_entries if entry.get('status') == 'success')
            error_count = len(log_entries) - success_count

            print(f"\nTotal log entries: {len(log_entries)}")
            print(f"  Successful: {success_count}")
            if error_count > 0:
                print(f"  Errors: {error_count}")

            # Show recent entries
            print("\nRecent generations (last 5):")
            print("-" * 80)
            for entry in log_entries[-5:]:
                timestamp = entry.get('timestamp', 'Unknown')[:19]  # Truncate to date+time
                speaker = entry.get('speaker_id', 'Unknown')
                status = entry.get('status', 'Unknown')
                print(f"  {timestamp} | {speaker} | {status}")
        else:
            print("\n⚠ Generation log is empty.")
    except Exception as e:
        print(f"\n⚠ Could not read generation log: {e}")
else:
    print("\n⚠ Generation log not found.")
    print("  Logs will be created when you generate your first audio clip.")

print("\n" + "=" * 80)
print("DIRECTORY PATHS")
print("=" * 80)

print(f"\nBase directory: {base_dir}")
print(f"Speakers directory: {speakers_dir}")
print(f"Output directory: {output_dir}")
print(f"Logs directory: {logs_dir}")

# Check directory sizes
def get_dir_size(path):
    """Get total size of directory in MB."""
    total = 0
    try:
        for dirpath, dirnames, filenames in os.walk(path):
            for filename in filenames:
                filepath = os.path.join(dirpath, filename)
                if os.path.exists(filepath):
                    total += os.path.getsize(filepath)
    except:
        pass
    return total / (1024 ** 2)  # Convert to MB

print("\nDirectory Sizes:")
print("-" * 80)
for name, path in [('Speakers', speakers_dir), ('Output', output_dir), ('Logs', logs_dir)]:
    if os.path.exists(path):
        size_mb = get_dir_size(path)
        print(f"  {name}: {size_mb:.2f} MB")
    else:
        print(f"  {name}: Directory not found")

print("\n" + "=" * 80)
print("SUMMARY COMPLETE")
print("=" * 80)
print("\nYour VibeVoice + LibriTTS-R pipeline is ready to use!")
print("Registered speakers can be used for TTS inference via the UI in Cell 9.")


# Documentation & Troubleshooting

## Usage Instructions Summary

This notebook implements a complete voice cloning and TTS pipeline using VibeVoice and LibriTTS-R. Follow these steps:

1. **Run Cell 1**: Install all dependencies and clone VibeVoice repository
2. **Run Cell 2**: Mount Google Drive and create required directories
3. **Copy Speaker Data**: Use Cell 3 instructions (Option A) or Cell 4 widget (Option B) to upload speaker folders
4. **Run Cell 5**: Validate uploaded speakers (minimum 30 seconds audio required)
5. **Run Cell 6**: Copy validated speakers to speakers directory
6. **Run Cell 7**: Download and load VibeVoice model from HuggingFace
7. **Run Cell 8**: Register speakers (create profiles with reference audio)
8. **Run Cell 9**: Set up inference UI widgets
9. **Run Cell 10**: Connect TTS generation function
10. **Use UI**: Select speaker, enter text, adjust parameters, and generate speech!

## Troubleshooting

### GPU Errors

**Problem**: "CUDA not available" or "GPU out of memory"

**Solutions**:
- Ensure you're using a GPU runtime: `Runtime → Change runtime type → Hardware accelerator → GPU`
- Try T4 GPU first (most stable for Colab)
- If using A100, you can try the larger 7B model (modify Cell 7)
- Restart runtime and clear outputs if memory is fragmented
- Reduce batch size or text length if memory errors occur during inference

### Missing Packages

**Problem**: Import errors or "Module not found"

**Solutions**:
- Re-run Cell 1 to reinstall dependencies
- Check that all pip installs completed successfully
- For VibeVoice-specific errors, ensure the repository was cloned correctly
- Try: `!pip install --upgrade <package_name>`

### Insufficient Drive Space

**Problem**: "No space left on device" or write failures

**Solutions**:
- Check your Google Drive storage: [drive.google.com](https://drive.google.com)
- Clean up staging directory: Remove processed speaker folders from `staging/`
- Remove old generated audio files from `output/`
- Use fewer speakers or smaller audio files
- Check available space using: `!df -h /content/drive/MyDrive`

### Corrupted Audio Files

**Problem**: Validation fails or audio files can't be loaded

**Solutions**:
- Check the validation report in `logs/validation_report_*.json`
- Re-upload corrupted speaker folders
- Ensure audio files are in supported formats: `.wav`, `.flac`, `.mp3`
- Check that files aren't corrupted during upload (verify file sizes)
- Use the `handle_corrupted_audio()` function from Cell 11

### Model Download Failures

**Problem**: "Error downloading model" from HuggingFace

**Solutions**:
- Check your internet connection
- Accept the model license on HuggingFace: [vibevoice/VibeVoice-1.5B](https://huggingface.co/vibevoice/VibeVoice-1.5B)
- Login to HuggingFace: `!huggingface-cli login`
- Model is cached in Drive, so re-downloads are skipped if already present
- Try manual download and place in `models/VibeVoice-1.5B/`

### Drive Mount Failures

**Problem**: "Failed to mount Google Drive"

**Solutions**:
- Re-run Cell 2 (it has automatic retry logic)
- Check that you authorized the mount request
- Ensure you have sufficient Drive storage
- Try: `drive.mount('/content/drive', force_remount=True)`

### Inference Errors

**Problem**: "Error generating speech" or no audio output

**Solutions**:
- Ensure speaker is registered (has `profile.json`)
- Check that reference audio files exist and are valid
- Verify VibeVoice model is loaded correctly
- Check GPU memory: `!nvidia-smi`
- Review error messages in the output area
- Check generation log in `logs/generation_log.csv`

### Slow Performance

**Problem**: Inference takes too long

**Solutions**:
- Use shorter text inputs for faster generation
- VibeVoice-1.5B is recommended for Colab (faster than 7B)
- Ensure you're using GPU (not CPU)
- Close other Colab tabs to free up resources
- First inference is slower (model loading), subsequent calls are faster

## Legal & Ethical Considerations

### Voice Cloning Ethics

**Important**: This notebook is for educational and research purposes. Please follow these guidelines:

- **Consent**: Only clone voices you have explicit permission to use
- **Public Figures**: Do not clone voices of public figures, celebrities, or politicians without permission
- **Personal Data**: Respect privacy - do not clone voices without consent
- **Misuse**: Do not use voice cloning for:
  - Impersonation or fraud
  - Creating misleading content
  - Harassment or defamation
  - Any illegal activities

### Dataset License

- **LibriTTS-R**: Derived from LibriSpeech, which is public domain
- **Commercial Use**: Check the specific LibriTTS-R license before commercial use
- **Attribution**: When using LibriTTS-R data, provide appropriate attribution

### VibeVoice License

- **VibeVoice**: MIT License (community fork)
- **Models**: Check HuggingFace model cards for specific model licenses
- **Usage**: Follow VibeVoice repository license terms

## Performance Notes

### GPU Memory Usage

- **VibeVoice-1.5B**: ~4-6 GB GPU memory during inference
- **VibeVoice-7B**: ~12-16 GB GPU memory (may not fit in Colab T4)
- **Recommendation**: Use 1.5B model for Colab (T4, A100, L4)

### Runtime Recommendations

| GPU Type | Model | Inference Speed | Notes |
|----------|-------|-----------------|-------|
| T4 | 1.5B | ~2-5 sec/word | Recommended for Colab |
| A100 | 1.5B | ~1-2 sec/word | Fast, but may have availability limits |
| A100 | 7B | ~3-5 sec/word | Only if you have A100 access |
| L4 | 1.5B | ~2-4 sec/word | Good alternative to T4 |

### Embedding Extraction vs. Full Adaptation

- **Embedding Extraction** (Current Implementation):
  - Fast: ~1-2 minutes per speaker
  - Uses reference audio for voice cloning
  - Good for quick setup and testing
  - Quality: Very good for most use cases

- **Full Fine-tuning** (Advanced):
  - Slow: ~30-60 minutes per speaker
  - Requires more audio data (5+ minutes recommended)
  - Better quality and consistency
  - Requires more GPU memory and time
  - Not implemented in this notebook (see VibeVoice repository for fine-tuning code)

### Expected Processing Times

- **Speaker Validation**: ~1-5 seconds per speaker (depends on audio file count)
- **Speaker Registration**: ~1-2 minutes per speaker
- **TTS Generation**: ~5-30 seconds per generation (depends on text length)
- **Model Loading**: ~2-5 minutes (first time only, cached afterwards)

## Additional Resources

- **VibeVoice Repository**: [github.com/vibevoice-community/VibeVoice](https://github.com/vibevoice-community/VibeVoice)
- **VibeVoice HuggingFace**: [huggingface.co/vibevoice](https://huggingface.co/vibevoice)
- **LibriTTS-R Dataset**: [openslr.org/141](https://www.openslr.org/141/)
- **VibeVoice Discord**: [discord.gg/ZDEYTTRxWG](https://discord.gg/ZDEYTTRxWG)

## Support

If you encounter issues not covered here:

1. Check the error messages carefully - they often contain helpful information
2. Review the logs in `logs/` directory
3. Verify all cells ran successfully in order
4. Check VibeVoice repository issues and discussions
5. Join the VibeVoice Discord community for help

---

**Note**: This notebook is a community implementation. The actual VibeVoice API may evolve, so some adjustments may be needed based on the latest VibeVoice repository version.
